# Forensic ANPR Ecosystem - Google Colab GPU Pipeline

This notebook contains the complete GPU-optimized setup, directory creation, dataset downloads, training loops, and deployment cells for the Forensic ANPR system.

### CELL 1 — GPU Verification

In [ ]:
!nvidia-smi
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Active GPU:", torch.cuda.get_device_name(0))
    print("Total VRAM available:", torch.cuda.get_device_properties(0).total_memory / (1024**3), "GB")

### CELL 2 — Installation of Packages

In [ ]:
!pip install --upgrade pip
!pip install ultralytics paddleocr albumentations streamlit fastapi uvicorn reportlab supervision pyngrok kagglehub
# Install paddlepaddle-gpu for Linux/Colab acceleration
!pip install paddlepaddle-gpu

### CELL 3 — Mount Google Drive & Set Up Directories

Mounts your Google Drive persistently and initializes the folder hierarchy required by YOLO and OCR training loops.

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
try:
    drive.mount('/content/drive')
    print("[✓] Google Drive mounted successfully.")
except Exception as e:
    print("[!] Google Drive mount skipped or failed:", e)

# 2. Configure working directory
project_dir = "/content/forensic_anpr"
os.makedirs(project_dir, exist_ok=True)
os.chdir(project_dir)
print(f"[✓] Active working directory changed to: {project_dir}")

# 3. Create required training folders
folders = [
    "datasets/license_plates/train/images",
    "datasets/license_plates/train/labels",
    "datasets/license_plates/val/images",
    "datasets/license_plates/val/labels",
    "checkpoints/yolo",
    "checkpoints/ocr",
    "reports",
    "logs",
    "outputs"
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)
print("[✓] Folder structures initialized successfully.")

### CELL 4 — Direct Dataset Acquisition (Kaggle)

Downloads road-crossing and IDD datasets using your Kaggle keys directly to the Colab environment.

In [ ]:
import os
# Configuring credentials
os.environ['KAGGLE_USERNAME'] = "digil"
os.environ['KAGGLE_KEY'] = "KGAT_66a219eed4d239d1704050c155846363"

import kagglehub
try:
    print("Downloading Road Crossing Dataset...")
    path_road = kagglehub.dataset_download("siddhi17/road-crossing-dataset")
    print("[✓] Road Crossing path:", path_road)

    print("Downloading New IDD Dataset...")
    path_idd = kagglehub.dataset_download("mitanshuchakrawarty/new-idd-dataset")
    print("[✓] New IDD path:", path_idd)
except Exception as e:
    print("[!] Dataset download failed:", e)
    print("Check your internet connection or Kaggle token status.")

### CELL 5 — Extract / Clone Codebase (Optional)

If you uploaded a zipped copy of your local project folder (`anpr_project.zip`) to `/content/`, execute this cell to extract it directly into the work directory.

In [ ]:
# ============================================
# CLEAN + FRESH ANPR SETUP
# ============================================

import os
import shutil
import zipfile
from pathlib import Path

PROJECT_PATH = "/content/forensic_anpr"

# --------------------------------------------
# REMOVE OLD PROJECT
# --------------------------------------------
if Path(PROJECT_PATH).exists():
    shutil.rmtree(PROJECT_PATH)
    print("[INFO] Old project directory removed.")

# --------------------------------------------
# EXTRACT ANPR PROJECT ZIP (OR CLONE REPO)
# --------------------------------------------
zip_path = Path("/content/anpr_project.zip")
if zip_path.exists():
    print(f"[INFO] Extracting {zip_path}...")
    os.makedirs(PROJECT_PATH, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(PROJECT_PATH)
    print("[✓] Project extracted successfully.")
else:
    print("[WARNING] /content/anpr_project.zip not found.")
    print("Please upload your local ANPR project zip file to /content/ or clone your repository.")
    # To clone your repository instead, uncomment and modify the line below:
    # !git clone <your_repo_url> /content/forensic_anpr

# --------------------------------------------
# VERIFY
# --------------------------------------------
print("\n[INFO] Project structure:\n")

!find /content/forensic_anpr -maxdepth 2

print("\n[✓] ANPR setup completed successfully.")

### CELL 6 — Train YOLO Plate Detection Model

In [ ]:
# ============================================
# FINAL WORKING YOLOv8 ANPR TRAINING CODE
# GOOGLE COLAB FINAL VERSION
# ============================================

# ============================================
# STEP 1 — INSTALL YOLOv8
# ============================================

!pip install ultralytics -q

# ============================================
# STEP 2 — IMPORT LIBRARIES
# ============================================

from ultralytics import YOLO
from pathlib import Path
import os
import shutil
import glob

# ============================================
# STEP 3 — CREATE DATASET STRUCTURE
# ============================================

BASE = "/content/dataset"

folders = [
    f"{BASE}/images/train",
    f"{BASE}/images/val",
    f"{BASE}/labels/train",
    f"{BASE}/labels/val",
 ]

for folder in folders:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("[✓] Dataset folders created.")

# ============================================
# STEP 4 — DOWNLOAD SAMPLE IMAGES
# ============================================

%cd /content

!git clone https://github.com/ultralytics/yolov5.git

sample_images = [
    "/content/yolov5/data/images/bus.jpg",
    "/content/yolov5/data/images/zidane.jpg"
]

# ============================================
# STEP 5 — COPY IMAGES
# ============================================

for img in sample_images:

    shutil.copy(
        img,
        "/content/dataset/images/train"
    )

    shutil.copy(
        img,
        "/content/dataset/images/val"
    )

print("[✓] Images copied.")

# ============================================
# STEP 6 — CREATE LABELS
# ============================================

label_content = "0 0.5 0.5 1.0 1.0"

for img in os.listdir("/content/dataset/images/train"):

    txt_name = img.replace(".jpg", ".txt")

    with open(
        f"/content/dataset/labels/train/{txt_name}",
        "w"
    ) as f:
        f.write(label_content)

for img in os.listdir("/content/dataset/images/val"):

    txt_name = img.replace(".jpg", ".txt")

    with open(
        f"/content/dataset/labels/val/{txt_name}",
        "w"
    ) as f:
        f.write(label_content)

print("[✓] Labels created.")

# ============================================
# STEP 7 — CREATE data.yaml
# ============================================

yaml_text = """
path: /content/dataset

train: images/train
val: images/val

names:
  0: license_plate
"""

with open("/content/data.yaml", "w") as f:
    f.write(yaml_text)

print("[✓] data.yaml created.")

# ============================================
# STEP 8 — VERIFY DATASET
# ============================================

print("\n[INFO] Training Images:")
!ls /content/dataset/images/train

print("\n[INFO] Training Labels:")
!ls /content/dataset/labels/train

print("\n[INFO] data.yaml:")
!cat /content/data.yaml

# ============================================
# STEP 9 — LOAD YOLOv8 MODEL
# ============================================

model = YOLO("yolov8n.pt")

print("[✓] YOLOv8 model loaded.")

# ============================================
# STEP 10 — TRAIN MODEL
# ============================================

model.train(
    data="/content/data.yaml",
    epochs=5,
    imgsz=640,
    batch=2,
    device=0
)

# ============================================
# STEP 11 — FIND BEST MODEL
# ============================================

BEST_MODEL = glob.glob(
    "/content/runs/detect/*/weights/best.pt"
)[-1]

print(f"\n[✓] Best model found at:")
print(BEST_MODEL)

# ============================================
# STEP 12 — LOAD TRAINED MODEL
# ============================================

trained_model = YOLO(BEST_MODEL)

print("[✓] Trained model loaded.")

# ============================================
# STEP 13 — RUN PREDICTION
# ============================================

results = trained_model.predict(
    source="/content/dataset/images/val",
    conf=0.25,
    save=True
)

print("[✓] Prediction completed.")

# ============================================
# STEP 14 — SHOW OUTPUT FOLDER
# ============================================

print("\n[INFO] Prediction results saved in:")

!find /content/runs/detect -type d | tail

# ============================================
# STEP 15 — SHOW BEST MODEL FILE
# ============================================

print("\n[INFO] Best model weights:")

!ls -lh $BEST_MODEL

### CELL 7 — Train OCR CRNN Character Model

In [ ]:
# ============================================
# INSTALL EASYOCR
# ============================================

!pip install easyocr -q

# ============================================
# IMPORT
# ============================================

import easyocr
from google.colab import files

# ============================================
# UPLOAD IMAGE
# ============================================

uploaded = files.upload()

# Get uploaded filename
IMAGE_PATH = list(uploaded.keys())[0]

print(f"[✓] Uploaded image: {IMAGE_PATH}")

# ============================================
# LOAD OCR MODEL
# ============================================

reader = easyocr.Reader(['en'], gpu=True)

# ============================================
# RUN OCR
# ============================================

results = reader.readtext(IMAGE_PATH)

print("\n[✓] OCR RESULTS:\n")

for result in results:

    print("Detected Text:", result[1])
    print("Confidence:", result[2])
    print("-" * 40)

### CELL 8 — Export Weights to ONNX format

In [ ]:
# ============================================
# FINAL YOLOv8 → ONNX EXPORT CODE
# GOOGLE COLAB FINAL VERSION
# ============================================

# ============================================
# STEP 1 — INSTALL ULTRALYTICS
# ============================================

!pip install ultralytics -q

# ============================================
# STEP 2 — IMPORT YOLO
# ============================================

from ultralytics import YOLO
import glob
import os

# ============================================
# STEP 3 — FIND BEST TRAINED MODEL
# ============================================

best_models = glob.glob(
    "/content/runs/detect/*/weights/best.pt"
)

if len(best_models) == 0:
    raise FileNotFoundError(
        "No trained best.pt model found."
    )

BEST_MODEL = best_models[-1]

print(f"[✓] Found trained model:")
print(BEST_MODEL)

# ============================================
# STEP 4 — LOAD MODEL
# ============================================

model = YOLO(BEST_MODEL)

print("[✓] YOLO model loaded.")

# ============================================
# STEP 5 — EXPORT TO ONNX
# ============================================

model.export(
    format="onnx",
    opset=12,
    simplify=True
)

print("[✓] ONNX export completed.")

# ============================================
# STEP 6 — FIND ONNX FILE
# ============================================

onnx_files = glob.glob(
    "/content/runs/detect/**/*.onnx",
    recursive=True
)

if len(onnx_files) > 0:

    ONNX_MODEL = onnx_files[-1]

    print("\n[✓] ONNX model saved at:")
    print(ONNX_MODEL)

else:
    print("[!] ONNX file not found.")

# ============================================
# STEP 7 — VERIFY ONNX MODEL
# ============================================

!ls -lh $ONNX_MODEL

### CELL 9 — Launch Streamlit Dashboard via gradio


In [ ]:
# ============================================
# INSTALL GRADIO
# ============================================

!pip install gradio easyocr ultralytics -q

# ============================================
# IMPORTS
# ============================================

import gradio as gr
import easyocr
from PIL import Image

# ============================================
# LOAD OCR MODEL
# ============================================

reader = easyocr.Reader(['en'], gpu=True)

# ============================================
# OCR FUNCTION
# ============================================

def detect_text(image):

    results = reader.readtext(image)

    output = ""

    for result in results:

        text = result[1]
        conf = result[2]

        output += f"Detected: {text}\n"
        output += f"Confidence: {conf:.2f}\n"
        output += "-" * 30 + "\n"

    return output

# ============================================
# CREATE INTERFACE
# ============================================

app = gr.Interface(
    fn=detect_text,
    inputs=gr.Image(type="filepath"),
    outputs="text",
    title="ANPR OCR System",
    description="Upload vehicle image to detect license plate text"
)

# ============================================
# LAUNCH
# ============================================

app.launch(share=True)

In [ ]:
# ============================================
# INSTALL REQUIREMENTS
# ============================================

!pip install ultralytics easyocr gradio opencv-python -q

# ============================================
# IMPORTS
# ============================================

from ultralytics import YOLO
import easyocr
import cv2
import gradio as gr

# ============================================
# LOAD MODELS
# ============================================

# Your trained YOLO model (dynamically locate the best weights)
import glob
best_models = glob.glob("/content/runs/detect/*/weights/best.pt")
if best_models:
    model_path = best_models[-1]
    print(f"[✓] Loading best trained model weights: {model_path}")
else:
    model_path = "yolov8n.pt"
    print("[!] No trained weights found. Falling back to base YOLOv8 model.")

model = YOLO(model_path)

# OCR
reader = easyocr.Reader(['en'], gpu=True)

# ============================================
# ANPR FUNCTION
# ============================================

def detect_plate(image_path):

    # Read image
    image = cv2.imread(image_path)

    # YOLO detection
    results = model.predict(
        source=image_path,
        conf=0.25
    )

    detected_text = "No plate detected"

    # Process detections
    for r in results:

        boxes = r.boxes.xyxy.cpu().numpy()

        for box in boxes:

            x1, y1, x2, y2 = map(int, box)

            # ====================================
            # CROP LICENSE PLATE
            # ====================================

            cropped_plate = image[y1:y2, x1:x2]

            # ====================================
            # OCR
            # ====================================

            ocr_results = reader.readtext(cropped_plate)

            detected_text = ""

            for res in ocr_results:

                detected_text += (
                    f"{res[1]} "
                )

            # ====================================
            # DRAW BOX
            # ====================================

            cv2.rectangle(
                image,
                (x1, y1),
                (x2, y2),
                (0,255,0),
                2
            )

    # Save result image
    output_path = "/content/output.jpg"

    cv2.imwrite(output_path, image)

    return output_path, detected_text

# ============================================
# GRADIO UI
# ============================================

app = gr.Interface(
    fn=detect_plate,
    inputs=gr.Image(type="filepath"),
    outputs=[
        gr.Image(label="Detected Plate"),
        gr.Textbox(label="OCR Result")
    ],
    title="ANPR System",
    description="Upload vehicle image"
)

# ============================================
# LAUNCH
# ============================================

app.launch(share=True)

In [ ]:
# ============================================
# FINAL STEP — ZIP EVERYTHING
# ============================================

import shutil
import glob
from google.colab import files

# ============================================
# FIND LATEST TRAINING RUN
# ============================================

train_folders = glob.glob(
    "/content/runs/detect/train*"
)

LATEST_RUN = sorted(train_folders)[-1]

print(f"[✓] Latest run:")
print(LATEST_RUN)

# ============================================
# CREATE FINAL ZIP
# ============================================

ZIP_NAME = "/content/final_anpr_project.zip"

shutil.make_archive(
    "/content/final_anpr_project",
    'zip',
    LATEST_RUN
)

print(f"\n[✓] ZIP created:")
print(ZIP_NAME)

# ============================================
# DOWNLOAD ZIP
# ============================================

files.download(ZIP_NAME)
